In [ ]:
import requests
import json
import openai

from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display


In [ ]:
import httpx
import asyncio

In [ ]:
MODEL = "llama3.2"
openai = openai.OpenAI(base_url="http://127.0.0.1:11434/v1", api_key="ollama")

In [ ]:
from typing import List

In [ ]:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url: str):
        self.url = url
        self.session = httpx.AsyncClient(headers=headers, verify=False)
        self.processed_urls = set()

    async def _fetch_page(self, url:str) -> str|None:

        if url in self.processed_urls:
            return None
        
        self.processed_urls.add(url)

        try:
            async with self.session.get(url) as response:
                response.raise_for_status()  # Raise an exception for bad status codes
            # response = await self.session.get(url, headers=headers, verify=False)
            self.body = await response.text()
            return self.body
        except Exception as e:
            raise str(e)
    
    async def extract_links(self) -> List[str]:
        self.body = await self._fetch_page(self.url)
        
        soup = BeautifulSoup(self.body, 'html.parser')

        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"
    
    async def close(self):
        await self.session.aclose()
        

In [ ]:
async def main():
    website_url = "https://www.languagecentre.ir/oproducts" 
    
    website_scraper = Website(website_url)
    
    try:
        print(f"Extracting links from: {website_url}")
        # فراخوانی متد async extract_links
        extracted_links = await website_scraper.extract_links() 
        
        if extracted_links:
            print(f"Successfully extracted {len(extracted_links)} unique links.")
            print("First 5 links:", extracted_links[:5])
            
            
            print("\n--- Website Details ---")
            print(website_scraper.get_contents())
        else:
            print("Could not extract any links or content.")
            
    finally:
        await website_scraper.close()

if __name__ == "__main__":
    asyncio.run(main())

In [ ]:
eng_books = Website("https://www.languagecentre.ir/oproducts")
eng_books.extract_links().links

In [ ]:
print(eng_books.get_contents())

In [ ]:
system_prompt = "You are provided with a list of links found on a webpage. \
            You are able to decide which of the links would be most relevant to include in a brochure about the company, \
            such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
system_prompt += "You should respond in JSON as in these examples:"
system_prompt += """Example1:
            {
                "links": [
                    {"type": "about page", "url": "https://full.url/goes/here/about"},
                    {"type": "careers page": "url": "https://another.full.url/careers"}
                ]
            }
        """
system_prompt += """Example2:
            {
                "links": [
                    {"type": "examle2 page", "url": "https://example2.urlformat/something/about/in/this/website"},
                    {"type": "more info page": "url": "https://example2.urlformat/something/else/here"}
                ]
            }         
        """
system_prompt += """Example3:
            {
                "links": [
                    {"type": "twitter page", "url": "https://twitter.com/twitter"},
                    {"type": "linkedin page": "url": "https://www.linkedin.com/company/linkedin_page"}
                ]
            }         
        """

system_prompt += """Example4:
            {
                "links": [
                    {"type": "youtube page", "url": "https://www.youtube.com/@youtube_channel"},
                    {"type": "instagram page": "url": "https://www.instagram.com/instagram_page/"}
                ]
            }         
        """
system_prompt += """Example5:
            {
                "links": [
                    {"type": "examle5 page", "url": "https://example5.urlformat/something/about/in/this/website"},
                    {"type": "more info page": "url": "https://example5.urlformat/something/else/here"}
                ]
            }         
        """

system_prompt += "IMPORTANT: Each 'url' field must contain a SINGLE URL string, not a list or array."

In [ ]:
def get_links_user_prompt(website, website_links):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, \
                    respond with the full https URL in JSON format. \
                    Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website_links)
    return user_prompt

In [ ]:
print(get_links_user_prompt(eng_books))

In [ ]:
async def get_links(url):
    website = Website(url)

    try:
        website_links = await website.extract_links()
        user_prompt = get_links_user_prompt(website, website_links)
        response = await openai.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
        ],
            response_format={"type": "json_object"}  # even if by defining response_format we have to specify an example in system prompt too
        )
        result = response.choices[0].message.content
        return json.loads(result)
    
    finally:
        await website.close()

In [ ]:
get_links("https://lastsecond.ir/blog/10745-iran-local-foods")

In [ ]:
get_links("https://lastsecond.ir/")

In [ ]:
import re
from urllib.parse import urlparse

def clean_url(url):
    """Clean and validate URL string."""
    if not url or not isinstance(url, str):
        return None
    
    # If it looks like a string representation of a list
    if url.startswith('[') and url.endswith(']'):
        # Extract first URL from the list
        match = re.search(r"'(https?://[^']+)'", url)
        if match:
            return match.group(1)
        return None
    
    # Basic URL validation
    parsed = urlparse(url)
    if parsed.scheme and parsed.netloc:
        return url
    
    return None

In [ ]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        link_type = link.get("type") or link.get("Type") or ""
        raw_url = link.get("url")
        # Clean the URL
        clean_url_str = clean_url(raw_url)
        if not clean_url_str:
            print(f"Warning: Invalid URL for {link_type}: {raw_url}")
            result += f"\n\n{link_type}\n[Unable to fetch: Invalid URL]"
            continue

        try:
            result += f"\n\n{link_type}\n"
            result += Website(link["url"]).get_contents()
        except Exception as e:
            print(f"Error fetching {clean_url_str}: {e}")
            result += f"\n\n{link_type}\n[Error loading page: {str(e)}]"
                
    return result

In [ ]:
get_all_details("https://lastsecond.ir/pages/aboutus")

In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("Language Center", "https://www.languagecentre.ir/oproducts")

In [ ]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
# create_brochure("Language Center", "https://www.languagecentre.ir/oproducts")

In [ ]:
# create_brochure("lingano", "https://lingano.com/")

In [ ]:
create_brochure("Recepies", "https://lastsecond.ir/blog/10745-iran-local-foods")